In [0]:
%load_ext autoreload
%autoreload 2
# Enables autoreload; learn more at https://docs.databricks.com/en/files/workspace-modules.html#autoreload-for-python-modules
# To disable autoreload; run %autoreload 0

In [0]:
from pyspark.sql import SparkSession

from src.ingestion.detect_delimiter import detect_delimiter
from src.ingestion.detect_header import detect_header
from src.ingestion.parse_file import parse_file
from src.transformation.create_dataframe import create_dataframe
from src.processing.clean_text import clean_text
from src.processing.extract_skills import extract_skills
from src.matching.find_overlap import find_overlap
from pyspark.sql.functions import col, udf, array, explode
from pyspark.sql.types import ArrayType
from pyspark.sql.types import StructType, StructField, StringType, IntegerType

spark = SparkSession.builder.appName("pipeline_runner").getOrCreate()

profile_raw = [
    "Generated by HR System",
    "Export Date: 2026-06-20",
    "Department: Data Analytics",
    "",
    "profile_id;name;experience;skills;;;;",
    "1;John Doe;Built ETL pipelines with Spark and Airflow;python,sql,spark;;;;",
    "2;Alice Smith;Worked on SQL dashboards and Power BI;sql,excel,powerbi;;;;",
    "3;Bob Lee;Developed ML models using Python and Pandas;python,pandas,machine learning;;;;",
    "4;Sara Khan;Built data lake pipelines in Databricks;spark,databricks,airflow;;;;"
]

job_raw = [
    "Job Export System",
    "Generated: 2026-06-20",
    "",
    "job_id;title;skills;;;;",
    "101;Data Engineer;python,sql,spark,airflow;;;;",
    "102;BI Analyst;sql,excel,tableau;;;;",
    "103;ML Engineer;python,machine learning,pandas;;;;",
    "104;Platform Engineer;spark,databricks,deltalake;;;;"
]

def filter_metadata_(profile_raw):
    return [
        l for l in profile_raw
        if ";" in l or "|" in l or "," in l
    ]


clean_lines_profile = filter_metadata(profile_raw)
clean_lines_jobs = filter_metadata(job_raw)


delimiter_result_profile = detect_delimiter(clean_lines_profile, [',', ';', '\t', '|'])
delimiter_result_jobs = detect_delimiter(clean_lines_jobs, [',', ';', '\t', '|'])


delimiter_profile = delimiter_result_profile["result"]["delimiter"]
delimiter_jobs = delimiter_result_jobs["result"]["delimiter"]




header_result_profile = detect_header(clean_lines_profile, delimiter_profile)
header_result_jobs = detect_header(clean_lines, delimiter_jobs)


parsed_result_profile = parse_file(clean_lines, delimiter_result, header_result_profile)
parsed_result_jobs = parse_file(clean_lines, delimiter_result, header_result_jobs)


records_profile = parsed_result_profile["result"]["data"]
records_jobs = parsed_result_jobs["result"]["data"]

for r in records_profile:
    if "profile_id" in r:
        r["profile_id"] = int(r["profile_id"])
    else:
        print("Missing profile_id row:", r)

for r in records_jobs:
    if "job_id" in r:
        r["job_id"] = int(r["job_id"])
    else:
        print("Missing job_id row:", r)

schema = StructType([
    StructField('profile_id', IntegerType(), True),
    StructField('name', StringType(), True),
    StructField('experience', StringType(), True)
])
df_result_profile = create_dataframe(records_profile, spark, schema=schema)
df_result_jobs = create_dataframe(records_jobs, spark, schema=schema)

df_profile = df_result_profile["result"]["dataframe"]
df_jobs = df_result_jobs["result"]["dataframe"]

#display claeaned dataframe
cleaned_result_profile = clean_text(df, column="experience")
cleaned_result_jobs = clean_text(df, column="experience")

cleaned_profile_df = cleaned_result_profile["result"]["dataframe"]
cleaned_job_df = cleaned_result_jobs["result"]["dataframe"]

# extract skills from experience column
profile_result = extract_skills(cleaned_profile_df, "experience")
jobs_result = extract_skills(cleaned_job_df, "experience")

# create new dataframes with skills column
profile_df = profile_result["result"]["dataframe"]
jobs_df = jobs_result["result"]["dataframe"]

# display schema of dataframes to see datatype of skills column for profile and jobs
#display(profile_df.printSchema())
#display(jobs_df.printSchema())

# select skills column for profile and jobs
profile_skills = profile_df.select("skills")
jobs_skills = jobs_df.select("skills")

# display skills column
display(profile_skills)
display(jobs_skills)

over_lap_results = find_overlap(profile_skills, jobs_skills)

profile_result = extract_skills(cleaned_profile_df, "experience")
jobs_result = extract_skills(cleaned_job_df, "experience")

#display(over_lap_df)

